In [87]:
import pandas as pd
import matplotlib as plt
import numpy as np
from functools import reduce
import seaborn as sns

# Load data into dfs

In [59]:
hockey_scouting_notes = pd.read_csv('../data/hockey_scouting_notes.csv')   # joinable by international_id
contracts_competition = pd.read_pickle('../data/contracts_competition.pkl') # joinable by international_id
performance = pd.read_csv('../data/performance.tsv', sep='\t') # joinable by international_id

identity_card_0 = pd.read_csv('../data/identity_card_0.tsv', sep='\t', header=0, names=['international_id',
                                                                                'medical_id',
                                                                                'first_name',
                                                                                'last_name',
                                                                                'gender',
                                                                                'age',
                                                                                'birth_city',
                                                                                'nationality'])
identity_card_1 = pd.read_csv('../data/identity_card_1.csv')

medical_information = pd.read_excel('../data/medical_information.xlsx',
                                    skiprows=2,
                                    header=0,
                                    names=['medical_id',
                                           'height',
                                           'weight',
                                           'age_in_years',
                                           'shoe_size',
                                           'body_fat_percentage',
                                           'fitness_level',
                                           'sprint_time',
                                           'medical_information',
                                           'return_date',
                                           'physician_signature']).drop(columns=['shoe_size',
                                                                                 'return_date',
                                                                                 'physician_signature']) # joinable by medical_id

moms_notes = pd.read_json('../data/moms_notes.json')


# Data cleaning and processing

## Identity card

In [4]:
def transfer_roman_to_int(roman_numeral):
    if pd.isna(roman_numeral) or not isinstance(roman_numeral, str):
        return roman_numeral

    roman_map = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}
    total = 0
    prev_value = 0
    for char in reversed(roman_numeral.upper().strip()):
        value = roman_map.get(char, 0)
        if value < prev_value:
            total -= value
        else:
            total += value
        prev_value = value
    return total

identity_card_0['international_id']=identity_card_0['international_id'].apply(transfer_roman_to_int)


In [5]:
# Take out the ius from last names an "us" from "male" first names
identity_card_0['last_name'] = identity_card_0['last_name'].apply(lambda x: x[:-3])
identity_card_0['first_name'] = identity_card_0.apply(lambda x: x['first_name'][:-2] if x['gender'] == 'male' else x['first_name'],axis=1)

In [6]:
identity_card = pd.concat([identity_card_0,identity_card_1])

# international_id

# gender -> normalize
gender_map = {'f':'female',
              'female': 'female',
              
              'm': 'male',
              'male': 'male',
              
              'other': 'other'}

identity_card['gender'] = identity_card['gender'].map(gender_map)

# birth_city and nationality -> normalize 
identity_card['birth_city'] = identity_card['birth_city'].str.lower().str.strip()
identity_card['nationality'] = identity_card['nationality'].str.lower().str.strip()

#drop birth city
identity_card = identity_card.drop(columns = ['birth_city'])

#drop people older than 122
identity_card=identity_card[identity_card['age'] <= 122]
identity_card


,international_id,medical_id,first_name,last_name,gender,age,nationality
0,275,039e0ab7-e36d-4bc7-a0b1-3e93e8ec73e9,Keith,Allee,male,23.0,austria
1,2560,070dfb40-76c9-4894-90e5-7463d060d49b,Alyssa,Kohl,female,31.0,austria
2,2323,fa06ed07-6f09-4ec2-9f11-a70917307270,Emily,Belanger,female,29.0,france
3,7708,59f94605-e953-4117-b9ee-09ba1392e895,Stefanie,Fiedler,female,25.0,italy
5,9337,e59f1d26-859b-4f06-b12f-3815f886db36,Brittany,Meeker,female,28.0,switzerland
...,...,...,...,...,...,...,...
9107,2660,94d60877-0aee-4318-aa6d-3deae0cb78fa,Walker,Rice,male,25.0,usa
9108,4550,26d12241-ad53-418b-9b07-a0bacbf24213,Barbara,Ramsey,female,23.0,norway
9109,6981,d2ad5fdd-6f48-4395-b901-bd107f0ed85c,David,Pawlicki,male,18.0,usa
9110,1249,b8822ca3-be3c-49a5-91be-ab701f562f8e,Sharon,Hancock,female,26.0,germany


### ID card cleanup actions and remarks

* International Ids from id_card_0 transfered from roman to arab numerals
* From id_card_0 take out "ius" in last name
* From id_card_0 take out "us" in first names of males (One guys name was Hilarious before and now its only Hilario, which is a bit weird but i don't even know with this dataset) :D
* Normalize gender mapping
* Normalize cities and countries to lowercase

Remarks
* Some age outliers, what to do with that??? diverse and synthesized dataset, leave old people in
* Birth cities and nationalities often don't match
* There are 37 random rows with no data apart from international_id and medical_id, I'd drop those.

TODO
* Drop brith city because nationality and birth city doesnt match, not important
* drop 37 random rows
* drop people older than the oldest person on earth

## Scouting notes

In [7]:
# fill nan in scout_notes with no notes
hockey_scouting_notes['scout_notes'] = hockey_scouting_notes['scout_notes'].fillna('no notes')

#fill nan values in dominant_hand with none
hockey_scouting_notes['dominant_hand'] = hockey_scouting_notes['dominant_hand'].fillna('none')

#drop people with 0 years + not rookie
rows = hockey_scouting_notes[(hockey_scouting_notes['years_played'] == 0) & (hockey_scouting_notes['experience_level'] != 'rookie')].index
hockey_scouting_notes.drop(rows, inplace=True)

#drop people who have more year played pro than years played
hockey_scouting_notes = hockey_scouting_notes[hockey_scouting_notes['years_played'] >= hockey_scouting_notes['years_pro']]
hockey_scouting_notes

,international_id,position,dominant_hand,experience_level,years_played,years_pro,scout_notes
0,2991,defense,left,veteran,8.0,8.0,"effort varies throughout the game, awkward str..."
3,6229,left wing,left,rookie,1.0,1.0,no notes
4,1303,defense,none,veteran,9.0,9.0,"poor awareness in defensive zone, strong and a..."
6,2571,left wing,none,veteran,5.0,5.0,"celebrates every goal like it wins the cup, ra..."
7,7353,defense,none,veteran,2.0,2.0,a player with a high hockey IQ who reads the g...
...,...,...,...,...,...,...,...
9995,6627,left wing,right,sophomore,1.0,1.0,refuses to change stick after a goal
9996,5202,defense,left,veteran,10.0,5.0,sets the tone early in games
9997,2182,goalie,left,veteran,1.0,1.0,combines intelligence with strong leadership q...
9998,3387,defense,left,rookie,6.0,6.0,shoots from anywhere just in case


### Scouting notes cleanup actions and remarks

Remarks
* Again some missing values, would decide later based on the classification what to do with them
* There are 37 missing rows again, matching the ids of the ones before

TODOS
* delete 37 random rows
* fill NaN in scout_notes with "no notes"
* fill NaN in dominant_hand with "none"
* drop people with 0 years + veteran
* check if years_pro is more than years_played --> drop

## Contracts competition

In [8]:
contracts_competition

,international_id,contracts_signed,salary,captain,won_championship,jersey_number,draft_year,number_of_previous_teams
0,6032,5.0,669578.77,False,True,48.0,2022.0,12.0
1,3654,5.0,1647102.80,False,False,10.0,2013.0,15.0
2,8729,6.0,1678593.48,False,False,63.0,2016.0,14.0
3,2300,1.0,927858.79,False,False,46.0,2015.0,8.0
4,8407,2.0,710328.25,True,True,79.0,2012.0,15.0
...,...,...,...,...,...,...,...,...
9995,4579,5.0,490417.83,False,False,7.0,2020.0,14.0
9996,8411,4.0,3389293.11,True,False,95.0,2015.0,15.0
9997,8570,12.0,1221299.65,False,False,1.0,2010.0,10.0
9998,2010,6.0,2279861.54,False,False,3.0,2017.0,8.0


In [9]:
# Cleaning contracts_competition

# contracts_signed, jersey_number, draft_year, number_of_previous_teams -> convert to int
contracts_competition['contracts_signed'] = contracts_competition['contracts_signed'].astype("Int64")
contracts_competition['jersey_number'] = contracts_competition['jersey_number'].astype("Int64")
contracts_competition['draft_year'] = contracts_competition['draft_year'].astype("Int64")
contracts_competition['number_of_previous_teams'] = contracts_competition['number_of_previous_teams'].astype("Int64")

contracts_competition['number_of_previous_teams'] = contracts_competition['number_of_previous_teams'].fillna(0)

# jersey_number
contracts_competition

,international_id,contracts_signed,salary,captain,won_championship,jersey_number,draft_year,number_of_previous_teams
0,6032,5,669578.77,False,True,48,2022,12
1,3654,5,1647102.80,False,False,10,2013,15
2,8729,6,1678593.48,False,False,63,2016,14
3,2300,1,927858.79,False,False,46,2015,8
4,8407,2,710328.25,True,True,79,2012,15
...,...,...,...,...,...,...,...,...
9995,4579,5,490417.83,False,False,7,2020,14
9996,8411,4,3389293.11,True,False,95,2015,15
9997,8570,12,1221299.65,False,False,1,2010,10
9998,2010,6,2279861.54,False,False,3,2017,8


### Contracts, competition actions and remarks
* Change columns to int where it makes sense
* Fill number_of previous years to 0 where it was na. Assumption is that these are the players that have not played in another team that have nan in this column.

Remarks
* Again some missing values in contracts_signed and number_of_previous_teams, would decide later based on the classification what to do with them
* There are 37 missing rows again, matching the ids of the ones before
* Outliers in draft_year, probably corresponding with old players in the first dataset
* Some crazy high outliers in salaries too
* After contracts are resolved, also worth to check contracts against the number of prevoius teams

## Performance

In [10]:
# change value types

performance["goals"] = performance["goals"].astype("Int64")
performance["assists"] = performance["assists"].astype("Int64")
performance["num_of_shots"] = performance["num_of_shots"].astype("Int64")
performance["shot_attempts"] = performance["shot_attempts"].astype("Int64")
performance["high_danger_shots"] = performance["high_danger_shots"].astype("Int64")
performance["medium_danger_shots"] = performance["medium_danger_shots"].astype("Int64")
performance["low_danger_shots"] = performance["low_danger_shots"].astype("Int64")
performance["winning_goals"] = performance["winning_goals"].astype("Int64")
performance["power_play_goals"] = performance["power_play_goals"].astype("Int64")
performance["puck_touches"] = performance["puck_touches"].astype("Int64")
performance["puck_recoveries"] = performance["puck_recoveries"].astype("Int64")
# performance["penalties_taken"] = performance["penalties_taken"].astype("Int64")
performance["goals_against_total"] = performance["goals_against_total"].astype("Int64")
performance["passes_attempted"] = performance["passes_attempted"].astype("Int64")
performance["passes_completed"] = performance["passes_completed"].astype("Int64")
performance["games_missed_due_to_injury"] = performance["games_missed_due_to_injury"].astype("Int64")

In [11]:
# recalculate shooting percentage and save percentage to make more sense

performance['shooting_percentage'] = performance['goals']/performance['num_of_shots']*100
performance['save_percentage'] = performance['save_percentage'].apply(lambda x: 100 if x > 100 else x)
performance['penality_minutes'] = np.where(performance['penalties_taken'] > 0, performance['penality_minutes'], 0)

In [12]:
# change time columns to seconds

performance['puck_possession_time'] = performance['puck_possession_time']*60
performance['penalty_kill_time'] = performance['penalty_kill_time']*60
performance['power_play_time'] = performance['power_play_time']*60
performance['time_on_ice'] = performance['time_on_ice']*60*60


In [13]:
# drop columns which are inconsistent with other columns

performance.drop('puck_touches', axis=1, inplace=True)
performance.drop('time_between_penalties', axis=1, inplace=True)
performance.drop('goals_against_total', axis=1, inplace=True)

In [14]:
print(performance[performance['winning_goals'] > performance['goals']]['international_id'].count())
print(performance[performance['power_play_time'] > performance['time_on_ice']]['international_id'].count())
print(performance[performance['puck_possession_time'] > performance['time_on_ice']]['international_id'].count())
print(performance[performance['penalty_kill_time'] > performance['time_on_ice']]['international_id'].count())
print(performance[performance['power_play_goals'] > performance['goals']]['international_id'].count())

0
0
0
0
8630


In [15]:
performance = performance[performance['time_on_ice'] < 1000000]

In [16]:
performance

,international_id,goals,assists,num_of_shots,shot_speed,shot_attempts,shooting_percentage,high_danger_shots,medium_danger_shots,low_danger_shots,...,puck_recoveries,puck_possession_time,penalty_kill_time,penality_minutes,penalties_taken,goals_against_average,passes_attempted,passes_completed,pass_completion_rate,games_missed_due_to_injury
0,6938,166,244,461,130.6172,480,36.008677,158,185,137,...,142,614.4,193.8,0.0,0.0,0.00,759,148,19.50,28
1,3924,98,166,459,76.4679,490,21.350763,142,186,162,...,138,0.0,208.2,2.0,1.0,0.00,456,354,77.63,21
2,3365,87,167,439,82.1817,449,19.817768,134,165,150,...,78,1136.4,0.0,0.0,0.0,0.00,371,171,46.09,19
3,1284,93,155,447,73.0825,447,20.805369,141,136,170,...,296,0.0,108.6,16.0,1.0,0.00,411,137,33.33,17
4,5333,104,169,496,75.9673,503,20.967742,188,163,152,...,183,0.0,286.8,0.0,0.0,0.00,455,183,40.22,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1606,56,124,418,77.1258,460,13.397129,160,143,157,...,102,0.0,240.0,0.0,0.0,0.00,792,512,64.65,19
9996,9894,64,143,424,102.5506,433,15.09434,154,162,117,...,86,2280.6,326.4,1.0,3.0,0.00,436,404,92.66,15
9997,5548,66,120,447,88.0325,470,14.765101,157,182,131,...,173,0.0,64.2,24.0,1.0,0.00,388,167,43.04,17
9998,2370,97,280,450,118.9816,476,21.555556,140,195,141,...,452,0.0,0.0,0.0,0.0,19.27,726,340,46.83,30


### Performance actions and remarks

Remarks
* How is shooting_percentage calculated? The number doesn't make sense at all. -> Calculated properly
* danger shots add up to shot_attempts not num_of_shots. -> good to know
* There are some people with save_percentages higher than 100. -> set to 100
* There are some people with higher power_play_time than time_on_ice. Could be time_on_ice is in hours, where power_play is in seconds but seems weird to me. -> assumption: time_on_ice in hours, power_play_time
* Puck touches again don't make any sense, as 453 players have more goals than puck touches and 5932 have more shot attempts than puck touches. -> drop puck touches
* Puck recoveries also don't make any sense as there are 2650 players with more recoveries than puck touches... -> drop puck_touches
* 3922 players have higher puck_possesion time than time_on_ice -> assume puck_posession is minutes and change it to seconds
* 841 player have a higher penalty kill time than time_on_ice... -> change penalty kill time to seconds from minutes
* 5762 players with 0 penalties taken and non-0 penalty time. -> set penalty minutes to 0 when player doesn't have any penalties
* 5941 players with 0 penalties and non-0 time_between_penalties. -> drop time_between penalties, because it doesn't make sense, and is probably not too relevant
* 1314 players with higher goals agains on average than total goals against. -> drop_goals_against total, as the average is more important.
* 6513 players with higher passes attempted than puck touches -> puck touches dropoped
* 2 players with time on ice bigger than 1milion seconds, outliers -> removed
* Number of shots generally around 450, some lower outliers, maybe goalies
* More than 8000 players have more powerplay_goals than goals, this could be

TODO
* normalizing save_percentages (max 100)
* assume that time_on_ice is in hours, and power_play_time is minutes and normalize into seconds
* drop puck touches

DONE HERE:
* Calculate shooting percentage properly instead of the nonsense
* Set max save_percentage to 100 from those who had more than 100
* Change time_on_ice from hours to seconds and power play time from minutes, puck_posession from minutes too
* dropped puck_touches, because it collided with many other columns
* Changed penalty kill time to seconds from minutes assuming
* droppend time_between_penalties column
* dropped 37 nan values

## Medical info

In [61]:
medical_information.dropna(subset=['weight','sprint_time'], inplace=True)

In [62]:
medical_information['height'] = medical_information['height'].apply(lambda x: x[:-2] if x[-2:] == 'cm' else x)
medical_information['height'] = medical_information['height'].apply(lambda x: float(x[:-1])*100 if x[-1:] == 'm' else x)

In [72]:

# convert age to int
medical_information['age_in_years'] = medical_information['age_in_years'].astype("Int64")

# normalize fitness level and create map with scores from 1-5, 0 for nans
medical_information['fitness_level'] = medical_information['fitness_level'].str.lower().str.strip()
medical_information['fitness_level'] = medical_information['fitness_level'].fillna('no notes')

fitness_map = {'nan': 0,
               'no notes': 0,
               
               'poor': 1,
               'average': 2,
               'good': 3,
               'excellent': 4,
               'elite': 5}

medical_information['fitness_level'] = medical_information['fitness_level'].map(fitness_map)

# remove percent symbol in body_fat_percentage 
if medical_information['body_fat_percentage'].dtype == 'object':
    medical_information['body_fat_percentage'] = (
        medical_information['body_fat_percentage']
        .str.replace('%', '', regex=False)
        .astype(float) / 100
    )


# normalize height and weight?
medical_information['height'] = medical_information['height'].astype('float').round()

if medical_information['weight'].dtype == 'object':
    medical_information['weight'] = (
        medical_information['weight']
        .str.replace('kg','', regex=False)).astype('float').round()
    
# fill nans in medical_information
medical_information['medical_information'] = medical_information['medical_information'].fillna('no notes')




In [73]:
medical_information

,medical_id,height,weight,age_in_years,body_fat_percentage,fitness_level,sprint_time,medical_information
0,368bcbfc-cc90-43da-841f-aa57d4d789c7,179.0,82.0,24,0.2087,2,3.9727,no notes
1,f7164dc2-6646-420e-b6b5-42fc07c11f6d,180.0,94.0,29,0.2338,0,4.1549,no notes
2,09c9de13-5d3a-4b75-a7ec-ce10bb32efdd,163.0,79.0,25,0.2341,3,4.0825,no notes
3,be8969bf-6de2-49fd-b0e9-c014bdb970d6,182.0,81.0,29,0.2128,3,3.5001,no notes
4,476cd0f0-0154-4e70-be7c-661007e7fcc4,168.0,109.0,25,0.3171,4,3.7619,no notes
...,...,...,...,...,...,...,...,...
9995,d8686224-e95b-4da5-86a9-0f43a4f63f26,175.0,90.0,21,0.2489,4,4.0910,no notes
9996,6d9916ac-307f-4ceb-8f7d-a284b76831bc,160.0,84.0,20,0.2830,5,4.3003,no notes
9997,bf144ec6-af6e-4a4f-9026-7f02999205d7,159.0,78.0,28,0.2562,2,3.8575,no notes
9998,9ba4ccea-771d-4cfc-9172-d2bf922178f3,174.0,80.0,31,0.2519,5,3.6665,no notes


### Medical info actions and remarks

Done
* Dropped the 37 NaN players
* Dropped players with no sprint_time recorded
* turn age to int
* create fitness map to have numerical scores
* remove percentage sign in body fat and turn them to floats
* normalize height and weight
* fill nans in medical information notes

### Merge Datasets

In [86]:
# merge identity_cards and medical information, since they are the only ones that share medical_id
identity_medical = pd.merge(identity_card, medical_information, on='medical_id')

# create collection of relevant datasets
hockey_data = [identity_medical,hockey_scouting_notes,performance,contracts_competition]

#merge them on international_id
hockey_data_merged = reduce(lambda  left,right: pd.merge(left,right,on=['international_id'], how='outer'), hockey_data)

hockey_data_merged


,international_id,medical_id,first_name,last_name,gender,age,nationality,height,weight,age_in_years,...,passes_completed,pass_completion_rate,games_missed_due_to_injury,contracts_signed,salary,captain,won_championship,jersey_number,draft_year,number_of_previous_teams
0,1,368bcbfc-cc90-43da-841f-aa57d4d789c7,Christopher,Ortega,male,24.0,usa,179.0,82.0,24,...,222.0,38.95,21.0,8.0,854228.19,False,False,59.0,2019.0,6.0
1,2,f7164dc2-6646-420e-b6b5-42fc07c11f6d,Brett,White,male,29.0,netherlands,180.0,94.0,29,...,240.0,61.54,14.0,1.0,162467.83,False,False,26.0,2014.0,9.0
2,3,09c9de13-5d3a-4b75-a7ec-ce10bb32efdd,Cecil,Garcia,male,25.0,latvia,163.0,79.0,25,...,492.0,75.00,23.0,8.0,1472259.21,False,False,89.0,2018.0,12.0
3,4,be8969bf-6de2-49fd-b0e9-c014bdb970d6,Patricia,Connelly,female,29.0,norway,182.0,81.0,29,...,545.0,57.19,20.0,9.0,3588858.90,False,False,4.0,2014.0,7.0
4,5,476cd0f0-0154-4e70-be7c-661007e7fcc4,Linda,Davis,female,25.0,czechia,168.0,109.0,25,...,281.0,69.38,16.0,4.0,1958583.53,False,False,47.0,2018.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,d8686224-e95b-4da5-86a9-0f43a4f63f26,Michelle,Young,female,21.0,denmark,175.0,90.0,21,...,319.0,75.95,19.0,8.0,646136.08,False,False,9.0,2022.0,6.0
9996,9997,6d9916ac-307f-4ceb-8f7d-a284b76831bc,Megan,Allison,female,20.0,russia,160.0,84.0,20,...,366.0,80.44,25.0,1.0,499351.30,False,False,87.0,2023.0,4.0
9997,9998,bf144ec6-af6e-4a4f-9026-7f02999205d7,James,Hoffpavir,male,28.0,switzerland,159.0,78.0,28,...,375.0,48.45,21.0,3.0,1190363.62,True,False,4.0,2015.0,15.0
9998,9999,9ba4ccea-771d-4cfc-9172-d2bf922178f3,Liana,Googe,female,31.0,russia,174.0,80.0,31,...,587.0,94.68,25.0,9.0,1614528.25,True,False,42.0,2012.0,15.0


### Visualizations